## Okta 개요

Okta는 기업용 보안 ID 솔루션을 제공하는 클라우드 기반 ID 및 액세스 관리 서비스로, 애플리케이션과 서비스 전반에서 원활한 인증 및 권한 부여를 지원합니다.

주요 기능:
* **Single Sign-On(SSO)**: 한 번 인증하여 여러 애플리케이션에 액세스
* **Multi-Factor Authentication(MFA)**: 추가 검증 방법을 통한 보안 강화
* **Adaptive Authentication**: 사용자 행동과 context를 기반으로 하는 위험 기반 인증 정책
* **Universal Directory**: 중앙 집중식 사용자 관리 및 프로필 동기화
* **API Access Management**: API 보안을 위한 OAuth 2.0 및 OpenID Connect 지원

## 학습 목표
Okta를 AgentCore Identity의 IdP로 사용하여 사용자를 인증하고, 에이전트가 사용자를 대신해 보호된 리소스에 액세스하도록 권한을 부여할 수 있습니다. 이 Notebook에서는 사용자가 에이전트를 호출하기 전에 인증하는 Okta 기반 Inbound Auth를 살펴봅니다.

## Authorization Code Flow
OAuth 2.0 authorization code flow는 웹 애플리케이션에서 사용자를 안전하게 인증하고 access token을 얻기 위한 권장 방식입니다. 이 흐름은 다음 단계로 구성됩니다.
1. 인증을 위해 사용자를 Okta로 리디렉션
2. 로그인 성공 후 authorization code 수신
3. code를 access token 및 refresh token으로 교환
4. 토큰을 사용해 보호된 리소스에 액세스

이 통합 pattern을 사용하면 애플리케이션에 안전한 표준 기반 인증을 유지하면서 AgentCore에서 Okta의 강력한 ID 관리 기능을 활용할 수 있습니다.

## 튜토리얼 아키텍처

```
┌──────────┐  1. Credentials  ┌──────────┐  2. JWT Token  ┌──────────┐
│  Client  │ ───────────────► │   Okta   │ ─────────────► │  Client  │
└──────────┘                  └──────────┘                └──────────┘
                                                                 │
                                                                 │ 3. Bearer JWT
                                                                 ▼
┌──────────┐                  ┌──────────┐                ┌──────────┐
│  Client  │ ◄─────────────── │ Bedrock  │ ─────────────► │  Agent   │
└──────────┘  6. Response     │AgentCore │  4. Invoke     └──────────┘
                              └──────────┘                      │
                                    ▲                           │
                                    └─────── 5. Response ───────┘
```

<figure>
    <img src="images/16.png">
</figure>

## 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| 에이전트 유형       | 단일                                                                             |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.5                                                     |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, Strands Agent 및 Amazon Bedrock 모델 사용  |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 쉬움                                                                             |
| Inbound Auth        | Okta                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                     |

### 주요 기능

* Okta 기반 Inbound Auth를 적용해 Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Amazon Bedrock 모델 사용
* Strands Agents 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* 새 IAM 역할, 정책, 사용자를 생성할 IAM 권한
* 새 AgentCore Agent를 생성할 IAM 권한
* Okta 계정
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

## 학습 목표 1: AgentCore에서 사용할 Okta 설정

## 1단계: Okta IdP 설정

Okta 인증으로 AgentCore Runtime을 구성하려면 먼저 Okta를 IdP로 설정해야 합니다. 이 섹션에서는 Okta tenant 생성, 애플리케이션 구성, 필요한 사용자 및 claim 설정 과정을 안내합니다.

### 1.1 Okta Developer 계정 생성

Okta 계정이 없다면 https://developer.okta.com/signup/ 으로 이동하여 "Sign up for Integrator Free Plan"을 선택하고 가입하세요.

### 1.2 테스트 사용자 추가

1. Okta 계정에 로그인합니다.
2. **Directory**, **People**을 차례로 선택하고 **Add person**을 클릭합니다.

   <figure>
       <img src="images/9.png">
   </figure>

3. 양식을 작성합니다.
   - **Activation**에서 **Activate now**를 선택합니다.
   - **I will set password**를 선택하고 사용자의 비밀번호를 설정합니다.
   - **User must change password on first login**을 선택 해제합니다.
   - **Save**를 클릭합니다.

   <figure>
       <img src="images/10.png">
   </figure>

### 1.3 Application Integration 생성

4. **Applications**를 선택한 다음 **Create App Integration**을 클릭합니다.

   <figure>
       <img src="images/1.png">
   </figure>

5. sign-in method로 **OIDC - OpenID Connect**를 선택하고 애플리케이션 유형으로 **Web Application**을 선택합니다.

   <figure>
       <img src="images/2_enhanced.png">
   </figure>

6. 애플리케이션을 구성합니다.
   - App integration name에 **AgentCore Inbound Auth**를 입력합니다.
   - grant type으로 **Authorization Code**를 선택합니다.
   - 에이전트를 실행할 리전에 따라 redirect URL로 "https://bedrock-agentcore.us-west-2.amazonaws.com/identities/oauth2/callback" 또는 "https://bedrock-agentcore.us-east-1.amazonaws.com/identities/oauth2/callback"을 사용합니다.

   <figure>
       <img src="images/3_enhanced.png">
   </figure>

   - assignments에서 **Allow everyone in your organization to access**를 선택하고 **Enable immediate access**는 선택된 상태로 둡니다. **Save**를 클릭합니다.

   <figure>
       <img src="images/5_enhanced.png">
   </figure>

   - 나중에 사용할 **Client ID**와 **Secret**을 복사합니다.

   <figure>
       <img src="images/6_enhanced.png">
   </figure>

### 1.4 Authorization Server 구성

7. 왼쪽 메뉴에서 **Security**, **API**를 차례로 선택하고 authorization server 이름을 클릭합니다.

   <figure>
       <img src="images/7_enhanced.png">
   </figure>

   - **Audience**를 복사하여 나중에 사용할 수 있도록 저장합니다.
     > **참고**: 이 예제에서는 기본 **Audience**를 변경했습니다. 다른 앱에 영향을 주지 않도록 audience를 변경하려면 새 authorization server를 추가하는 것이 좋습니다.
   
   - **Scopes**를 클릭하고 **agentcore**라는 새 scope를 추가합니다.

   <figure>
       <img src="images/agencore.png">
   </figure>

   - **Claims**를 클릭하고 다음 **client_id** 및 **scope** claim을 추가합니다.

   <figure>
       <img src="images/8.png">
   </figure>

### 1.5 구성 값 수집

Okta 설정을 완료하면 다음 값을 준비해야 합니다.

- **OKTA_CLIENT_ID**: General 탭의 애플리케이션 Client ID
- **OKTA_CLIENT_SECRET**: General 탭의 애플리케이션 Client Secret  
- **OKTA_AUDIENCE**: Audience(예: `testagentcore`)
- **OKTA_TOKEN_URL**: Okta domain + `/oauth2/default/v1/token`
- **OKTA_DISCOVERY_URL**: Okta domain + `/oauth2/default/.well-known/openid-configuration`

다음 단계에서 필요하므로 이 값을 준비해 두세요.

참고:
1. Okta는 AWS 서비스가 아닙니다. Okta 관련 비용은 Okta 문서를 참조하세요.
2. 다음 단계의 화면은 변경될 수 있습니다. Okta 애플리케이션 설정에 대한 최신 지침은 Okta 문서를 참조하세요.

## 학습 목표 2 - Okta Inbound Auth를 사용하는 간단한 에이전트 설정

#### 사전 요구 사항
1. 필수 패키지 설치
2. 패키지 import
3. Notebook 전반에서 사용할 account ID 확인
4. AWS 리전을 "us-west-2"로 설정. Bedrock AgentCore를 지원하는 모든 리전을 사용할 수 있으며 지원 리전은 https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html 참조

## 2단계: 환경 설정

먼저 개발 환경을 설정하고 필수 종속성을 설치하겠습니다.

In [ ]:
# 가상 환경 생성 및 활성화
!python -m venv .venv
!source .venv/bin/activate

이 코드를 실행하려면 Python 환경에 Strands Agents 모듈을 설치해야 합니다.

종속성 파일에 Strands Agents 모듈, AgentCore SDK, AgentCore starter toolkit을 추가하고 **requirements.txt**로 저장하세요.

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit
PyJWT

In [ ]:
# requirements.txt에서 필수 패키지 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# 모든 필수 패키지가 올바르게 설치되었는지 확인
try:
    import bedrock_agentcore
    import strands

    print("✅ All packages installed successfully")
    print("✅ bedrock-agentcore: imported")
    print("✅ strands: imported")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure all packages are installed correctly")

## 3단계: 환경 변수 구성


이 Notebook 전반에서 사용할 주요 정보의 환경 변수를 설정합니다.

In [ ]:
# Okta 구성 - 실제 값으로 교체
import os

os.environ["OKTA_CLIENT_ID"] = "YOUR_CLIENT_ID_VALUE"
os.environ["OKTA_CLIENT_SECRET"] = "YOUR_CLIENT_SECRET_VALUE"
os.environ["OKTA_AUDIENCE"] = "YOUR_OKTA_AUDIENCE"
os.environ["OKTA_TOKEN_URL"] = "https://your.okta.com/oauth2/default/v1/token"
os.environ["OKTA_DISCOVERY_URL"] = "https://your.okta.com/oauth2/default/.well-known/openid-configuration"

## 4단계: 에이전트 코드
이 Notebook의 핵심 학습 목표는 Okta를 사용하는 Inbound Auth이므로 에이전트는 간단하게 구성합니다.

In [ ]:
%%writefile simple_agent.py
import argparse, json
from strands import Agent, tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
agent = Agent()

@app.entrypoint
def invoke(payload):
    """인바운드 인증 데모용 간단한 에이전트 함수입니다."""
    user_message = payload.get("prompt", "Hello! How can I help you today?")
    
    # 사용 가능한 경우 세션 정보 가져오기
    session_id = payload.get("session_id", "no-session")
    
    # 세션을 인식하는 간단한 응답
    response = f"Hello! I'm a simple agent with session ID: {session_id}. You asked: {user_message}"
    
    result = agent(response)
    return {"result": result.message}

if __name__ == "__main__":
    app.run()

## 5단계: Okta 인증으로 AgentCore Runtime 구성

Okta OAuth 인증으로 AgentCore Runtime을 구성합니다.

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# 계정 ID와 리전 가져오기
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = "us-west-2"

print(f"Account ID: {account_id}")
print(f"Region: {region}")

# 환경 변수 사용
discovery_url = os.getenv("OKTA_DISCOVERY_URL")
client_id = os.getenv("OKTA_CLIENT_ID")
audience = os.getenv("OKTA_AUDIENCE")

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id[:4]}****{client_id[-4:] if client_id else 'None'}")  # 마스킹됨
print(f"Audience: {audience}")

agentcore_runtime = Runtime()

# OAuth 구성으로 시도
try:
    response = agentcore_runtime.configure(
        entrypoint="simple_agent.py",
        auto_create_execution_role=True,
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name="okta_inbound_auth_agent",
        authorizer_configuration={
            "customJWTAuthorizer": {
                "discoveryUrl": discovery_url,
                "allowedClients": [client_id],
                "allowedAudience": [audience],
            }
        },
    )
    print("✅ OAuth configuration successful")
except Exception as e:
    print(f"❌ OAuth configuration failed: {e}")

response

## 6단계: AgentCore Runtime에 에이전트 시작

에이전트 구성을 마쳤으므로 AgentCore Runtime에 시작하겠습니다.

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

## 7단계: AgentCore Runtime 상태 확인

에이전트가 준비될 때까지 배포 상태를 모니터링합니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

## 8단계: 테스트용 Agent ARN 저장

테스트에 사용할 Agent ARN을 추출하여 저장합니다.

테스트에 사용할 Agent ARN을 추출하여 저장합니다.

In [ ]:
# launch_result에서 ARN 추출
if hasattr(launch_result, "agent_arn") and launch_result.agent_arn:
    agent_arn = launch_result.agent_arn
    os.environ["AGENT_ARN"] = agent_arn
    print(f"📝 Agent ARN: {agent_arn}")
    print(f"📝 Agent ID: {launch_result.agent_id}")
    print(f"📝 ECR URI: {launch_result.ecr_uri}")
else:
    print("⚠️  Could not extract Agent ARN from launch result")
    print("Launch result:", launch_result)

# 배포 상태 확인
if status == "READY":
    print("✅ Agent deployed successfully and ready for testing!")
elif status in ["CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]:
    print(f"❌ Agent deployment failed with status: {status}")
else:
    print(f"⚠️  Unexpected status: {status}")

## 9단계: 테스트 Client 생성

Okta OAuth 흐름과 session 지원 에이전트 호출을 검증할 테스트 client를 생성합니다.

In [ ]:
import requests
import json
import time
import urllib.parse
import logging
import uuid

# 로깅 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# 구성
client_id = os.getenv("OKTA_CLIENT_ID")
client_secret = os.getenv("OKTA_CLIENT_SECRET")
audience = os.getenv("OKTA_AUDIENCE")
token_url = os.getenv("OKTA_TOKEN_URL")
AGENT_ARN = os.getenv("AGENT_ARN")

print("✅ Configuration loaded successfully")

Okta에서 OAuth access token을 가져오는 함수 정의

In [ ]:
def get_oauth_token():
    """Okta에서 OAuth 토큰을 가져옵니다."""
    data = {"grant_type": "client_credentials", "scope": "agentcore"}

    logger.info("🔐 Getting OAuth token...")

    response = requests.post(token_url, data=data, auth=(client_id, client_secret))

    response.raise_for_status()
    token_data = response.json()
    logger.info("✅ OAuth token obtained")
    return token_data["access_token"]

인증 및 session 지원으로 에이전트를 호출하는 함수 정의

In [ ]:
def invoke_agent(access_token, query, session_id=None):
    """세션 ID를 지원하는 에이전트를 호출합니다."""
    escaped_agent_arn = urllib.parse.quote(AGENT_ARN, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

    # 제공되지 않은 경우 세션 ID 생성
    if not session_id:
        session_id = f"okta-inbound-session-{int(time.time())}-{uuid.uuid4().hex[:8]}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    payload = {"prompt": query, "session_id": session_id}

    logger.info(f"🚀 Invoking agent with query: {query}")
    logger.info(f"📋 Session ID: {session_id}")

    response = requests.post(url, headers=headers, json=payload, timeout=300)
    response.raise_for_status()

    result = response.json()
    logger.info("✅ Agent response received")
    return result, session_id

### 인증되지 않은 요청 테스트(실패해야 함)

먼저 에이전트가 인증되지 않은 요청을 올바르게 거부하는지 확인합니다.

In [ ]:
# 인증되지 않은 요청 테스트 - 실패해야 함
print("=" * 50)
print("TEST: Unauthenticated Request (Should Fail)")
print("=" * 50)

try:
    escaped_agent_arn = urllib.parse.quote(AGENT_ARN, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

    headers = {
        "Content-Type": "application/json"
        # 참고: Authorization 헤더 없음
    }

    payload = {"prompt": "Hello without authentication"}

    response = requests.post(url, headers=headers, json=payload, timeout=30)
    print(f"❌ Unexpected success! Status: {response.status_code}")
    print(f"Response: {response.text}")

except requests.exceptions.HTTPError as e:
    print(f"✅ Expected authentication failure: {e.response.status_code}")
    print(f"Error message: {e.response.text}")
except Exception as e:
    print(f"✅ Expected authentication error: {e}")

print("\n🔒 This confirms that authentication is required!")

인증된 요청에 사용할 OAuth access token을 Okta에서 가져오기

In [ ]:
# OAuth token 가져오기
access_token = get_oauth_token()
print("✅ OAuth token obtained successfully")

session ID 추적을 사용한 인증된 에이전트 호출 테스트

In [ ]:
# 테스트 1: 세션 ID를 사용한 간단한 쿼리
print("=" * 50)
print("TEST 1: Simple Query with Session ID")
print("=" * 50)

result1, session_id1 = invoke_agent(
    access_token,
    "Hello can you guide me about AWS security best practices for authentication?",
)
print(f"Session ID used: {session_id1}")
print(json.dumps(result1, indent=2))

동일한 session ID를 재사용하여 session 연속성 테스트

In [ ]:
# 테스트 2: 동일한 세션 ID로 대화 계속하기
print("=" * 50)
print("TEST 2: Continue Conversation with Same Session")
print("=" * 50)

result2, session_id2 = invoke_agent(access_token, "What was my previous question?", session_id1)
print(f"Session ID used: {session_id2}")
print(json.dumps(result2, indent=2))

잘못된 scope 이름으로 권한 없는 액세스의 scope 검증 테스트

scope는 애플리케이션의 권한을 정의하며 OAuth2 권한 부여에 세분화된 액세스 제어를 제공합니다.

In [ ]:
import jwt


def check_token_scopes(access_token, required_scope="agentcore"):
    """토큰에 필수 범위가 있는지 확인합니다."""
    try:
        # 데모를 위해 검증 없이 token 디코딩(프로덕션에서는 서명 검증)
        decoded = jwt.decode(access_token, options={"verify_signature": False})
        token_scopes = decoded.get("scp", [])

        # token에 필수 scope가 있는지 확인
        has_access = required_scope in token_scopes

        return {
            "has_access": has_access,
            "token_scopes": token_scopes,
            "required_scope": required_scope,
        }
    except Exception as e:
        return {"error": str(e), "has_access": False}


# 테스트 3: Scope 검증 - 부정 시나리오
print("=" * 50)
print("TEST 3: Scope Validation - Negative Scenario")
print("=" * 50)

# 잘못된 scope 이름 확인 - 'admin' scope는 관리자 권한을 부여함
wrong_scope_check = check_token_scopes(access_token, "admin")
print(f"Wrong scope check result: {json.dumps(wrong_scope_check, indent=2)}")

if wrong_scope_check.get("has_access"):
    print("✅ Token has required scope")
else:
    print("❌ Access denied: Token does not have required scope")
    print(f"Token has scopes: {wrong_scope_check['token_scopes']}")
    print(f"Required scope: {wrong_scope_check['required_scope']}")
    print("Agent invocation would be blocked at application level")

올바른 scope 이름으로 권한 있는 액세스의 scope 검증 테스트

scope는 인증된 애플리케이션이 수행할 수 있는 작업을 제한하여 안전한 API 액세스를 지원합니다.

In [ ]:
# 테스트 4: Scope 검증 - 긍정 시나리오
print("=" * 50)
print("TEST 4: Scope Validation - Positive Scenario")
print("=" * 50)

# 올바른 scope 확인 - 'agentcore' scope는 AgentCore agent 호출 권한을 부여함
scope_check = check_token_scopes(access_token, "agentcore")
print(f"Scope check result: {json.dumps(scope_check, indent=2)}")

if scope_check.get("has_access"):
    print("✅ Token has required scope - proceeding with agent call")
    result4, session_id4 = invoke_agent(access_token, "I have the right scope! Tell me about AWS security.")
    print(f"Agent response: {result4}")
else:
    print("❌ Token lacks required scope - access denied")

## 마무리 및 정리
이 Notebook에서는 다음 내용을 알아보았습니다.
- OAuth Authorization Code flow를 제공하도록 Okta API와 애플리케이션 설정
- AgentCore Runtime을 생성하고 Okta Inbound Auth를 사용하는 에이전트 배포
- 토큰을 받아 보호된 에이전트 액세스에 사용
- session 관리 및 연속성 확인

#### 생성된 리소스

In [ ]:
# 생성된 agent ID 표시
if hasattr(launch_result, "agent_id"):
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"Agent ARN: {launch_result.agent_arn}")
else:
    print("Agent information not available")

#### AgentCore Runtime 삭제

이 튜토리얼에서 생성한 리소스를 정리합니다.

In [ ]:
# AgentCore Runtime 삭제
try:
    if hasattr(launch_result, "agent_id"):
        agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
        agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
        print(f"✅ Agent {launch_result.agent_id} deleted successfully")
    else:
        print("⚠️  No agent ID found to delete")
except Exception as e:
    print(f"❌ Error deleting agent: {e}")
    print("You may need to delete the agent manually from the AWS console")

## 마무리

이 Notebook에서는 다음 방법을 살펴보았습니다.

1. **Okta IdP 설정**: Okta tenant, 애플리케이션, authorization server 구성
2. **간단한 에이전트 생성**: session을 인식하는 기본 에이전트 구축
3. **OAuth 인증 구성**: Okta JWT 검증으로 AgentCore Runtime 설정
4. **인증을 적용한 배포**: Inbound Auth가 적용된 에이전트 배포
5. **인증 흐름 테스트**: OAuth 토큰 흐름 및 session 관리 검증

### 주요 학습 내용:

- **Inbound Auth**: Okta가 에이전트 endpoint를 보호하여 인증된 사용자만 에이전트를 호출하도록 보장
- **Session 관리**: 에이전트가 개인화된 응답에 session 정보를 활용
- **JWT 토큰 검증**: AgentCore가 Okta JWT 토큰을 자동으로 검증
- **보안**: 인증되지 않은 요청을 자동으로 거부

### 다음 단계:

- 사용자 기반 인증 흐름 구현
- 사용자 context를 활용하는 고급 에이전트 로직 추가
- 외부 API 액세스를 위한 Outbound Auth 살펴보기
- 추가 보안 계층을 위한 AgentCore Gateway 통합